In [5]:
# -*- coding: utf-8 -*-
# Instagram 캡션/해시태그 형태소 분석 (한글만) - 새 폴더(데이터셋/타임스탬프 단위) 저장

import os, re, ast, json, warnings
from datetime import datetime
from collections import Counter
import pandas as pd

# ===== 설정 =====
# 새로 올린 도시락 데이터로 기본값 설정 (필요 시 아래 주석처럼 교체)
INPUT_PATH  = "instagram_captions_샐러드_600.csv"
# INPUT_PATH  = "/mnt/data/instagram_captions_샐러드_600.csv"

# 결과 저장 루트(여기만 고정하면 데이터셋별/타임스탬프별로 자동 분리 저장)
OUTPUT_ROOT = "data"
MIN_COUNT   = 2

STAMP       = datetime.now().strftime("%Y%m%d_%H%M%S")
dataset_name = os.path.splitext(os.path.basename(INPUT_PATH))[0]
OUTPUT_DIR   = os.path.join(OUTPUT_ROOT, dataset_name, STAMP)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== 로딩 =====
def read_csv_safely(path):
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, encoding="utf-8-sig", errors="ignore")

df = read_csv_safely(INPUT_PATH)

# 컬럼 자동 탐색
cap_candidates = ["caption", "text", "content", "contents", "내용", "캡션", "post", "body", "설명"]
tag_candidates = ["hashtags", "tags", "hash_tags", "해시태그", "태그"]
def find_col(cands):
    for c in df.columns:
        if str(c).strip().lower() in [x.lower() for x in cands]:
            return c
    for c in df.columns:
        lc = str(c).strip().lower()
        if any(x.lower() in lc for x in cands):
            return c
    return None
CAP_COL = find_col(cap_candidates) or df.columns[0]
TAG_COL = find_col(tag_candidates)

print({"rows": len(df), "columns": list(df.columns), "CAP_COL": CAP_COL, "TAG_COL": TAG_COL})

# ===== 정규식 유틸 =====
url_re      = re.compile(r"(https?://\S+|www\.\S+)")
mention_re  = re.compile(r"@[A-Za-z0-9_\.]+")
email_re    = re.compile(r"[A-Za-z0-9\._%+\-]+@[A-Za-z0-9\.\-]+\.[A-Za-z]{2,}")
# 안전한 이모지 범위(한글 포함 구간 제거 X)
emoji_re    = re.compile("[" 
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
"]+")
# 일본어 제거
jp_re       = re.compile(r"[\u3040-\u309F\u30A0-\u30FF\uFF65-\uFF9F]")
# 한글만 남기기(영어/숫자/기타 제거, #는 해시태그 파싱 위해 유지)
non_korean_re = re.compile(r"[^ \t\n\r\u3131-\u318E\uAC00-\uD7A3#]")
# 한글 시퀀스(길이>=2)
korean_seq_re = re.compile(r"[\uAC00-\uD7A3\u3131-\u318E]{2,}")

def basic_clean_korean_only(s: str) -> str:
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = url_re.sub(" ", s)
    s = email_re.sub(" ", s)
    s = mention_re.sub(" ", s)
    s = emoji_re.sub(" ", s)
    s = jp_re.sub(" ", s)               # 일본어 제거
    s = non_korean_re.sub(" ", s)       # 영어/숫자/기타 제거 (한글 + # 유지)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ===== (가벼운) 한국어 불용어 =====
STOPWORDS = set("""
그리고 그러나 그런데 또한 또는 그래서 때문에 등의 즉 및 으로 로 은 는 이 가 을 를 과 와 하고 보다 에서 에게 에 에도 에는 에다가 으니까 면 도 만 까지 뿐 처럼 같은 듯 듯이 것 거 데 수 들 등 더 가장 제일 아주 매우 너무 정말 진짜 그냥 혹시 거의 대부분 여러 각각 모든 아무 이런 그런 저런 어떤 무슨 있다 없다 이다 아니다 하다 되다 같다
오늘 내일 지금맛있는 맛있는 있는 좋은 메뉴 오늘도 오늘은 많이 맛있게 먹고 요즘 가능 조금 오늘의 먹는 있어요 저는 가득 이렇게 바로 넣고
가능합니다 같이 그대로 듬뿍 살짝 맛을 좋아요 단체주문 모두 있습니다 하는 가지 먹으면 먹을 즐길 다들 입니다 단체 맛이 문의 오랜만에 이제 직접 다시 많은 시오픈입니다 없는 제가 감사합니다 넣어 다른 먹어도
먹었어요 약간 안녕하세요 역시 올려 남은 어제 올리고 주차 추천 맛있어요 먹은 보고 추가 맛있고 섞어 싶은 엄청 없이 이번 있어서
최소주문 같아요 구매 되는 미리 새로운 서울 소개합니다 수업 위해 이용 조합
테이블
합니다
해서
따로
맛있어서
매장
문의주세요
브랜드
사람
하나
해요
기본
맛있다
서비스가
이거
자주
최대한
향이
가격
그릇
더욱
메뉴가
메뉴로
부터
소스는
시식당
식사를
연락처
연휴
우리
위에
있도록
진행
프로필
필수
한번
협찬
곡성뚝방마켓
그래도
담아
맛의
부탁드립니다
않은
여주사랑카드
완전
정도
주말에
진한
해도
계속
기존
네이버
드디어
드세요
드셔보세요
따라
마음이
만들었는데
맞춰
먹는다
먼저
무료
분명
삶은
양이
위한
음식
잘라
제대로
주세요
지난
항상
ㅎㅎ
간단하게
갑자기
건강을
곡성마카롱
금요일
꾸준히
끼를
만나보세요
만드는
만들어봤어요
많아
많아서
맛은
먹으니
먹으러
메뉴는
언제나
언제든
엄마
있어
있으니
전용
전화문의

접시
제품
집에
챙겨
천도
추천합니다
특별한
팔로우
가능하니
구성으로
그런지
담은
되고
되어


""".split())

def tokenize_ko(text: str):
    toks = [w for w in korean_seq_re.findall(str(text)) if len(w) >= 2]
    return [w for w in toks if w not in STOPWORDS]

def parse_hashtags_korean(raw, fallback_text=""):
    if isinstance(raw, str) and raw.strip():
        s = raw
    elif isinstance(raw, list):
        s = " ".join(map(str, raw))
    else:
        s = str(fallback_text)
    s = basic_clean_korean_only(s)
    return tokenize_ko(s)

def clean_and_tokenize_caption_korean(s: str):
    s = basic_clean_korean_only(s)
    s_wo_hash = re.sub(r"#\S+", " ", s)     # 캡션 분석에서 해시태그 제거
    s_wo_hash = re.sub(r"\s+", " ", s_wo_hash).strip()
    return s, tokenize_ko(s_wo_hash)

# ===== 메인 처리 =====
cap_texts, cap_tokens_list, tag_tokens_list = [], [], []
for _, row in df.iterrows():
    cap_raw = row[CAP_COL] if CAP_COL in df.columns else ""
    tag_raw = row[TAG_COL] if TAG_COL and TAG_COL in df.columns else ""
    cap_cleaned, cap_toks = clean_and_tokenize_caption_korean(cap_raw)
    tag_toks = parse_hashtags_korean(tag_raw, fallback_text=cap_raw)
    cap_texts.append(cap_cleaned); cap_tokens_list.append(cap_toks); tag_tokens_list.append(tag_toks)

both_tokens_list = [ct + tt for ct, tt in zip(cap_tokens_list, tag_tokens_list)]

# ===== 빈도 집계 =====
def freq_df(token_lists, min_count=2):
    cnt = Counter()
    for toks in token_lists: cnt.update(toks)
    items = [(t, c) for t, c in cnt.items() if c >= min_count]
    items.sort(key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(items, columns=["token", "freq"])

cap_freq  = freq_df(cap_tokens_list, min_count=MIN_COUNT)
tag_freq  = freq_df(tag_tokens_list, min_count=MIN_COUNT)
all_freq  = freq_df(both_tokens_list, min_count=MIN_COUNT)

# ===== 저장 (새 폴더) =====
base = f"{dataset_name}"
cleaned_path = os.path.join(OUTPUT_DIR, f"{base}_ko_only_cleaned_{STAMP}.csv")
cap_freq_path = os.path.join(OUTPUT_DIR, f"{base}_ko_only_caption_freq_{STAMP}.csv")
tag_freq_path = os.path.join(OUTPUT_DIR, f"{base}_ko_only_hashtag_freq_{STAMP}.csv")
all_freq_path = os.path.join(OUTPUT_DIR, f"{base}_ko_only_all_freq_{STAMP}.csv")

df_out = df.copy()
df_out["ko_caption_clean"]   = cap_texts
df_out["ko_tokens_caption"]  = [json.dumps(t, ensure_ascii=False) for t in cap_tokens_list]
df_out["ko_tokens_hashtags"] = [json.dumps(t, ensure_ascii=False) for t in tag_tokens_list]
df_out["ko_tokens_all"]      = [json.dumps(t, ensure_ascii=False) for t in both_tokens_list]

df_out.to_csv(cleaned_path, index=False, encoding="utf-8-sig")
cap_freq.to_csv(cap_freq_path, index=False, encoding="utf-8-sig")
tag_freq.to_csv(tag_freq_path, index=False, encoding="utf-8-sig")
all_freq.to_csv(all_freq_path, index=False, encoding="utf-8-sig")

print("=== 저장 완료 ===")
print("저장 폴더:", OUTPUT_DIR)
print("정제:", cleaned_path)
print("캡션:", cap_freq_path)
print("해시태그:", tag_freq_path)
print("합본:", all_freq_path)
print({"caption_docs_with_tokens": sum(1 for t in cap_tokens_list if t),
       "hashtag_docs_with_tokens": sum(1 for t in tag_tokens_list if t),
       "both_docs_with_tokens": sum(1 for t in both_tokens_list if t)})

# 미리보기
print("\n[캡션 Top 30]\n", cap_freq.head(30))
print("\n[해시태그 Top 30]\n", tag_freq.head(30))
print("\n[합본 Top 30]\n", all_freq.head(30))


{'rows': 600, 'columns': ['url', 'caption', 'hashtags'], 'CAP_COL': 'caption', 'TAG_COL': 'hashtags'}
=== 저장 완료 ===
저장 폴더: data\instagram_captions_샐러드_600\20251024_161342
정제: data\instagram_captions_샐러드_600\20251024_161342\instagram_captions_샐러드_600_ko_only_cleaned_20251024_161342.csv
캡션: data\instagram_captions_샐러드_600\20251024_161342\instagram_captions_샐러드_600_ko_only_caption_freq_20251024_161342.csv
해시태그: data\instagram_captions_샐러드_600\20251024_161342\instagram_captions_샐러드_600_ko_only_hashtag_freq_20251024_161342.csv
합본: data\instagram_captions_샐러드_600\20251024_161342\instagram_captions_샐러드_600_ko_only_all_freq_20251024_161342.csv
{'caption_docs_with_tokens': 547, 'hashtag_docs_with_tokens': 596, 'both_docs_with_tokens': 600}

[캡션 Top 30]
     token  freq
0     샐러드   307
1   올리브오일    49
2      소금    48
3      매일    47
4    샌드위치    44
5     만들어    40
6     드레싱    39
7      후추    39
8     도시락    38
9      치즈    38
10    건강한    36
11     시간    36
12     함께    36
13    레몬즙    35
14   

In [6]:
# -*- coding: utf-8 -*-
"""
Instagram 캡션 형태소 상관/군집 분석 (도시락_1000)
- 입력: instagram_captions_도시락_1000_ko_only_cleaned_20251024_161231.csv
- 열: ko_tokens_all (문서별 형태소 리스트; 문자열 형태)
- 산출:
  1) morph_stats_도시락1000.csv      : 토큰별 df/tf/클러스터/중심성/랭킹
  2) morph_clusters_도시락1000.csv   : 클러스터별 Top 토큰 요약
  3) morph_edges_npmi_top_도시락1000.csv : 상위 NPMI 엣지
  4) npmi_heatmap_top25_도시락1000.png, top_pairs_npmi_bar_도시락1000.png
"""

import os, re, ast, math, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ================== 경로 설정 ==================
# ▶▶ 본인 PC 경로로 바꿔 사용하세요.
DATA_DIR = r"C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231"
FNAME   = "instagram_captions_도시락_1000_ko_only_cleaned_20251024_161231.csv"

IN_PATH  = Path(DATA_DIR) / FNAME
OUT_DIR  = Path(DATA_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ================== 파라미터 ==================
MIN_DF = 5          # 최소 문서수(등장 문서 기준) 필터
TOP_V  = 300        # 상위 DF 토큰만 사용(속도/안정성)
K_RANGE = (6, 12)   # K-means 군집 후보 범위(포함)
EMB_DIM = 80        # PPMI-SVD 임베딩 차원
MIN_CO = 5          # 엣지 저장 시 공동출현 최소 문서수
HEAT_TOP = 25       # 히트맵에 표시할 상위 토큰 수
PAIR_TOP = 20       # NPMI 상위 쌍 바차트 개수

# =============== 유틸 ===============
def parse_list(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except Exception: return []

def clean_tok(t):
    t = str(t).strip().replace(" ","")
    # 한글/영숫자만 유지
    return "".join(ch for ch in t if ch.isalnum() or ('가' <= ch <= '힣'))

def z(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

# =============== 1) 데이터 로드 & 토큰 전처리 ===============
df = pd.read_csv(IN_PATH, encoding="utf-8-sig")
docs_raw = df["ko_tokens_all"].apply(parse_list).tolist()

docs = []
for toks in docs_raw:
    toks2 = [clean_tok(t) for t in toks if isinstance(t,str)]
    toks2 = [t for t in toks2 if len(t) >= 2]
    docs.append(toks2)

N = len(docs)

# =============== 2) 어휘 선택(DF 기준) & DTM(이진) ===============
df_counter = Counter()
for toks in docs:
    for t in set(toks):
        df_counter[t] += 1

cand = [(t,c) for t,c in df_counter.items() if c >= MIN_DF]
cand.sort(key=lambda x: -x[1])
vocab = [t for t,_ in cand[:TOP_V]]
V = len(vocab)
vidx = {t:i for i,t in enumerate(vocab)}

# 희소 없이 빠른 벡터화(문서수 1000 수준 가정)
DTM = np.zeros((N, V), dtype=np.uint8)   # binary
tf  = np.zeros(V, dtype=np.int32)
for i, toks in enumerate(docs):
    cols = {vidx[t] for t in toks if t in vidx}
    if cols:
        DTM[i, list(cols)] = 1
        for j in cols:  # tf는 등장회수 기반으로 가려면 위에서 count, 여기선 문서내 unique만 셈
            pass
    # tf(출현 총횟수)는 문서내 중복 포함 계산
    for t in toks:
        if t in vidx: tf[vidx[t]] += 1

df_vec = DTM.sum(axis=0).astype(np.int32)

# =============== 3) 공동출현/확률/PMI/NPMI/PHI ===============
# co-occurrence: (VxV) = DTM^T * DTM
co = DTM.T @ DTM
total_docs = float(N)
p_t = df_vec / total_docs
p_xy = co / total_docs
EPS = 1e-12

# PMI/NPMI
PMI  = np.log((p_xy + EPS) / (p_t[:,None] * p_t[None,:] + EPS))
NPMI = PMI / (-np.log(p_xy + EPS))
np.fill_diagonal(PMI, 0.0)
np.fill_diagonal(NPMI, 0.0)

# Phi(이진 피어슨 상관)
n11 = co.astype(float)
n1_ = df_vec.astype(float)[:,None]
n_1 = df_vec.astype(float)[None,:]
n00 = N - (n1_ + n_1 - n11)
n10 = n1_ - n11
n01 = n_1 - n11
den = np.sqrt(n1_*(N-n1_)*n_1*(N-n_1)) + 1e-12
PHI = (n11*n00 - n10*n01) / den
np.fill_diagonal(PHI, 0.0)

# =============== 4) PPMI 임베딩 + K-means 군집 ===============
PPMI = PMI.copy()
PPMI[PPMI < 0] = 0.0

svd = TruncatedSVD(n_components=min(EMB_DIM, max(10, V-1)), random_state=42)
emb = svd.fit_transform(PPMI)

best = {"k": None, "sil": -1, "labels": None}
k_lo, k_hi = K_RANGE
for k in range(k_lo, k_hi+1):
    km = KMeans(n_clusters=k, n_init=8, random_state=42)
    labels = km.fit_predict(emb)
    # 계산량 줄이기 위해 표본 400개 한정
    try:
        sil = silhouette_score(emb, labels, metric="euclidean",
                               sample_size=min(400, len(emb)), random_state=42)
    except Exception:
        sil = -1
    if sil > best["sil"]:
        best = {"k": k, "sil": sil, "labels": labels}

labels = best["labels"]
k = best["k"]

# =============== 5) 클러스터 중심성 + 랭킹 ===============
NPMI_pos = NPMI.copy()
NPMI_pos[NPMI_pos < 0] = 0.0

cent = np.zeros(V)
for ci in range(k):
    idx = np.where(labels == ci)[0]
    if len(idx) == 0: continue
    sub = NPMI_pos[np.ix_(idx, idx)]
    cent[idx] = (sub.sum(axis=1) - np.diag(sub))

rank_score = z(df_vec) + z(cent)

token_stats = (
    pd.DataFrame({
        "token": vocab,
        "df": df_vec,
        "tf": tf,
        "cluster": labels,
        "cluster_centrality": cent,
        "rank_score": rank_score
    })
    .sort_values(["rank_score", "df"], ascending=[False, False])
    .reset_index(drop=True)
)

# 클러스터 요약
rows = []
for ci in range(k):
    sub = token_stats[token_stats["cluster"] == ci].head(15)
    rows.append({
        "cluster": ci,
        "size": int((labels == ci).sum()),
        "top_tokens": ", ".join(sub["token"].tolist())
    })
clusters_df = pd.DataFrame(rows).sort_values("cluster")

# =============== 6) 엣지(상위 NPMI) 표출 ===============
pairs = np.triu_indices(V, 1)
npmi_vals = NPMI[pairs]
co_vals = co[pairs]
mask = (npmi_vals > 0) & (co_vals >= MIN_CO)
order = np.argsort(-npmi_vals[mask])
take = min(4000, mask.sum())
i_idx = pairs[0][mask][order][:take]
j_idx = pairs[1][mask][order][:take]

edges_df = pd.DataFrame({
    "token1": [vocab[i] for i in i_idx],
    "token2": [vocab[j] for j in j_idx],
    "co_docs": co_vals[mask][order][:take].astype(int),
    "PMI": PMI[pairs][mask][order][:take],
    "NPMI": npmi_vals[mask][order][:take],
    "PHI": PHI[pairs][mask][order][:take]
})

# =============== 7) 저장 ===============
stats_path    = OUT_DIR / "morph_stats_도시락1000.csv"
clusters_path = OUT_DIR / "morph_clusters_도시락1000.csv"
edges_path    = OUT_DIR / "morph_edges_npmi_top_도시락1000.csv"

token_stats.to_csv(stats_path, index=False, encoding="utf-8-sig")
clusters_df.to_csv(clusters_path, index=False, encoding="utf-8-sig")
edges_df.to_csv(edges_path, index=False, encoding="utf-8-sig")

print(f"[DONE] k={k}, silhouette={best['sil']:.3f}")
print(f"- 토큰 통계 : {stats_path}")
print(f"- 클러스터  : {clusters_path}")
print(f"- 상위 엣지 : {edges_path}")

# =============== 8) 시각화(필요시 폰트 설정) ===============
# 히트맵 (상위 HEAT_TOP 토큰)
HEAT_TOP = min(HEAT_TOP, len(token_stats))
sel = token_stats["token"].head(HEAT_TOP).tolist()
sel_idx = [vocab.index(t) for t in sel]
mat = NPMI_pos[np.ix_(sel_idx, sel_idx)]

plt.figure(figsize=(9,7))
plt.imshow(mat, interpolation="nearest")
plt.xticks(range(HEAT_TOP), sel, rotation=90)
plt.yticks(range(HEAT_TOP), sel)
plt.title("NPMI Heatmap (Top tokens)")
plt.colorbar()
heat_path = OUT_DIR / "npmi_heatmap_top25_도시락1000.png"
plt.tight_layout(); plt.savefig(heat_path, dpi=200); plt.close()

# 상위 연관쌍 바차트
tp = edges_df.head(PAIR_TOP)
plt.figure(figsize=(10,7))
y = [f"{a}-{b}" for a,b in zip(tp["token1"], tp["token2"])]
plt.barh(range(len(y)), tp["NPMI"].values)
plt.yticks(range(len(y)), y)
plt.xlabel("NPMI")
plt.title("상위 형태소 연관쌍 (NPMI 기준)")
bar_path = OUT_DIR / "top_pairs_npmi_bar_도시락1000.png"
plt.tight_layout(); plt.savefig(bar_path, dpi=200); plt.close()

print(f"- 히트맵 PNG : {heat_path}")
print(f"- 상위쌍 PNG : {bar_path}")


[DONE] k=11, silhouette=0.103
- 토큰 통계 : C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231\morph_stats_도시락1000.csv
- 클러스터  : C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231\morph_clusters_도시락1000.csv
- 상위 엣지 : C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231\morph_edges_npmi_top_도시락1000.csv
- 히트맵 PNG : C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231\npmi_heatmap_top25_도시락1000.png
- 상위쌍 PNG : C:\Users\sagej\Digital_Pyhton_Study\인스타_트렌드_크롤링\도시락_1000\instagram_captions_도시락_1000\20251024_161231\top_pairs_npmi_bar_도시락1000.png
